# Additional Modeling Features

With the goal of improving model performance through additional feature engineering. A small subset of features were generated and stored in the S3 bucket to add signal for the modeling to identify.

In [22]:
import re
import sqlite3
import sys
from pathlib import Path

import pandas as pd
import requests

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    
from extract.config import settings
from extract.s3_manager import df_to_s3

CONN = sqlite3.connect(settings.db_path)

Some organizations have submitted filings several times within a single year. Although we don't know all of the reasons why, we do know that amendments can be make. Therefore, we opted to select only the most recent filing per intended year.

In [50]:
individual_filings = pd.read_sql(
     """
        select filing_id, ein, tax_year
        from (
            select
                *,
                row_number() over(PARTITION by ein, tax_year order by tax_period_end_date desc, return_timestamp desc) as rank
            from (
                select
                    tax_year,
                    tax_period_end_date,
                    return_timestamp,
                    filing_id,
                    ein
                from irs990_filings
                -- Our project only cares about public orgs, and did not extend into unrelated business expenses
                -- Some null exemption types are due to Trust filings, which was outside of the scope of our project
                where form_type not in ('990PF', '990T') and exempt_organization_type is not null
            ) f
        ) f2
        where f2.rank = 1
    """,
    CONN
)
assert len(individual_filings.loc[:, ['tax_year', 'ein']].drop_duplicates()) == len(individual_filings), "The previous pass did not create unique filings per tax year."
individual_filings.head()

,filing_id,ein,tax_year
0,4184533,010011694,2024
1,711148,010015091,2021
2,2146306,010015091,2022
3,2959539,010015091,2023
4,3732396,010015091,2024


In [51]:
features_base_q = """
select
	f.filing_id, f.ein, f.tax_year,
 	upper(f.filer_name) as current_filer_name, doing_business_as_name,
    form_type, exempt_organization_type,
	mission,
	case
		when total_revenue is not null and total_revenue <> 0 then grants_and_contributions / total_revenue
		else 0
	end as grants_as_rev_frac,
	case
		when voting_members_governing_body is not null and voting_members_governing_body <> 0 then voting_members_independent / voting_members_governing_body
		else null
	end as perc_ind_voting_members,
    political_activity_flag,
    total_expenses,
    case
    	when l.total_lobbying_expenditures_amt > l.fees_for_services_lobbying_amt then l.total_lobbying_expenditures_amt
		else l.fees_for_services_lobbying_amt
	end as lobbying_expenses,
    coalesce(filer_address_line1, '') || ' ' || coalesce(filer_address_line2, '') || ' ' || coalesce(filer_city, '') || ' ' || coalesce(filer_state, '') || ' ' || coalesce(filer_zip, filer_zip_code, '') as address
from irs990_filings f
	left join irs990_filing_lobbying l on f.filing_id = l.filing_id
-- Our project only cares about public orgs, and did not extend into unrelated business expenses
-- Some null exemption types are due to Trust filings, which was outside of the scope of our project
where f.form_type not in ('990PF', '990T') and exempt_organization_type is not null
"""

out_types = [
    'B', # Grant to related org
    'D', # Loan to related org
    'G', # Sale of assets to related org
    'J', # Lease of facilities, etc. to related org
    'L', # Performance of services, etc. for related org
    'P', # Reimbursement paid to related org
    'R'  # Other transfer of cash etc. to related org
]

in_types = [
    'C', # Grant from related org
    'E', # Loan from related org
    'H', # Purchase of assets from related org
    'K', # Lease of facilities, etc. from related org
    'M', # Performance of services, etc. from related org
    'Q', # Reimbursement paid from related org
    'S'  # Other transfer of cash etc. from related org
]

between_types = [
    'I', # Exhange of assets between related orgs
    'N', # Sharing of facilities, etc. between related orgs
    'O'  # Sharing of paid employees w. related orgs
]
org_transactions_base_q = f"""
select 
	filing_id, related_org_name,
	num_out_transactions + num_in_transactions + num_between_transactions as related_transaction_count,
	sum_out_transactions + sum_in_transactions + sum_between_transactions as related_transaction_sum
from (
	select 
		filing_id,
		case
			when type in ('{"','".join(out_types)}') then count(line_no)
			else null
		end as num_out_transactions,
		case
			when type in ('{"','".join(out_types)}') then sum(amount)
			else null
		end as sum_out_transactions
		--In
		case
			when type in ('{"','".join(in_types)}') then count(line_no)
			else null
		end as num_in_transactions,
		case
			when type in ('{"','".join(in_types)}') then sum(amount)
			else null
		end as sum_in_transactions
		-- Between
		case
			when type in ('{"','".join(between_types)}') then count(line_no)
			else null
		end as num_between_transactions,
		case
			when type in ('{"','".join(between_types)}') then sum(amount)
			else null
		end as sum_between_transactions
	from irs990_filing_related_org_transactions
	where
		type not in (
			'A', -- Receipts from controlled entity
			'F' -- Dividends from related org
		)
	group by filing_id
) org_transactions
"""
org_people_q = """
select 
	filing_id,
	count(line_no) as num_employees,
	--From Org
	avg(avg_weekly_hours_worked_org) as avg_avg_weekly_hours_org,
	avg(compensation_from_org) as avg_compensation_org,
	--From Related Org
	avg(avg_weekly_hours_worked_related_org) as avg_avg_weekly_hours_related_org,
	avg(compensation_from_related_org) as avg_compensation_related_org,
	--From Other
	avg(compensation_other) as avg_compensation_other
from irs990_filing_people
group by filing_id
"""

In [53]:
features_base_df = pd.read_sql(features_base_q, CONN)
features_base_df = features_base_df.merge(
    individual_filings.loc[:, ['filing_id']],
    on='filing_id', how='inner'
)
features_base_df.head()

,filing_id,ein,tax_year,current_filer_name,doing_business_as_name,form_type,exempt_organization_type,mission,grants_as_rev_frac,perc_ind_voting_members,political_activity_flag,total_expenses,lobbying_expenses,address
0,1,272292010,2019,NORTH CENTRAL ACADEMY,NaN,990,501(c)(3),MISSION IS TO IMPACT THE LIVES OF OUR STUDENTS...,0.673025,1.0,0,1137870.0,NaN,928 W MARKET STREET TIFFIN OH 44883
1,2,911152733,2020,SHEIKH ABDUL KADIR IDRESS MOSQUE TRUST,NaN,990,501(c)(2),"EXCLUSIVELY FOR RELIGIOUS, EDUCATIONAL AND SIM...",0.363716,0.0,0,5794.0,NaN,1420 NE NORTHGATE WAY SEATTLE WA 98125
2,3,223519265,2020,FRIENDS OF HOBOKEN CHARTER SCHOOL INC,NaN,990,501(c)(3),"TO FOSTER A QUALITY EDUCATION, FOSTER A NURTUR...",0.072674,1.0,0,25534.0,NaN,713 WASHINGTON STREET HOBOKEN NJ 07030
3,4,260741074,2020,NORTH CAROLINA TAX COLLECTORS ASSOCIATION,NaN,990EZ,501(c)(3),To provide continuing education and profession...,NaN,NaN,0,28827.0,NaN,65 Glen Road Box 246 Garner NC 27529
4,5,814522819,2020,CHUA PHAP NGHIEM INC,NaN,990EZ,501(c)(3),PLACE OF WORSHIP,NaN,NaN,0,85049.0,NaN,1610 TODDS LN Hampton VA 23666


Count the number of words present in the filer's mission statement. The idea is that a dark money org is more likely to have a shorter mission statement than a legitimate one.

In [54]:
features = features_base_df.copy()
features["mission"] = features.mission.str.split().str.len()
features.head()

,filing_id,ein,tax_year,current_filer_name,doing_business_as_name,form_type,exempt_organization_type,mission,grants_as_rev_frac,perc_ind_voting_members,political_activity_flag,total_expenses,lobbying_expenses,address
0,1,272292010,2019,NORTH CENTRAL ACADEMY,NaN,990,501(c)(3),32,0.673025,1.0,0,1137870.0,NaN,928 W MARKET STREET TIFFIN OH 44883
1,2,911152733,2020,SHEIKH ABDUL KADIR IDRESS MOSQUE TRUST,NaN,990,501(c)(2),10,0.363716,0.0,0,5794.0,NaN,1420 NE NORTHGATE WAY SEATTLE WA 98125
2,3,223519265,2020,FRIENDS OF HOBOKEN CHARTER SCHOOL INC,NaN,990,501(c)(3),41,0.072674,1.0,0,25534.0,NaN,713 WASHINGTON STREET HOBOKEN NJ 07030
3,4,260741074,2020,NORTH CAROLINA TAX COLLECTORS ASSOCIATION,NaN,990EZ,501(c)(3),26,NaN,NaN,0,28827.0,NaN,65 Glen Road Box 246 Garner NC 27529
4,5,814522819,2020,CHUA PHAP NGHIEM INC,NaN,990EZ,501(c)(3),3,NaN,NaN,0,85049.0,NaN,1610 TODDS LN Hampton VA 23666


The "doing business as name" (DBA) could be an opportunity for organizations to conceal their identity. Sometimes, organizations have multiple DBAs, but those are included in the Schedule O, which was outside of the scope of this project. Having more unique DBAs across time could be a way for a dark money organization to evade detection.

In [55]:
features = features.merge(
    features.groupby("ein",as_index=False).agg(dba_changes=("doing_business_as_name", "nunique")),
    on="ein",
    how="left"
)
features = features.drop("doing_business_as_name", axis=1)
features.head()

,filing_id,ein,tax_year,current_filer_name,form_type,exempt_organization_type,mission,grants_as_rev_frac,perc_ind_voting_members,political_activity_flag,total_expenses,lobbying_expenses,address,dba_changes
0,1,272292010,2019,NORTH CENTRAL ACADEMY,990,501(c)(3),32,0.673025,1.0,0,1137870.0,NaN,928 W MARKET STREET TIFFIN OH 44883,0
1,2,911152733,2020,SHEIKH ABDUL KADIR IDRESS MOSQUE TRUST,990,501(c)(2),10,0.363716,0.0,0,5794.0,NaN,1420 NE NORTHGATE WAY SEATTLE WA 98125,0
2,3,223519265,2020,FRIENDS OF HOBOKEN CHARTER SCHOOL INC,990,501(c)(3),41,0.072674,1.0,0,25534.0,NaN,713 WASHINGTON STREET HOBOKEN NJ 07030,0
3,4,260741074,2020,NORTH CAROLINA TAX COLLECTORS ASSOCIATION,990EZ,501(c)(3),26,NaN,NaN,0,28827.0,NaN,65 Glen Road Box 246 Garner NC 27529,0
4,5,814522819,2020,CHUA PHAP NGHIEM INC,990EZ,501(c)(3),3,NaN,NaN,0,85049.0,NaN,1610 TODDS LN Hampton VA 23666,0


Similarly, if the organization's name changes over time, it could also indicate that an org is trying to mask its identity.

In [56]:
features = features.merge(
    features.groupby("ein",as_index=False).agg(name_changes=("current_filer_name", "nunique")),
    on="ein",
    how="left"
)
features = features.drop("current_filer_name", axis=1)
features.head()

,filing_id,ein,tax_year,form_type,exempt_organization_type,mission,grants_as_rev_frac,perc_ind_voting_members,political_activity_flag,total_expenses,lobbying_expenses,address,dba_changes,name_changes
0,1,272292010,2019,990,501(c)(3),32,0.673025,1.0,0,1137870.0,NaN,928 W MARKET STREET TIFFIN OH 44883,0,1
1,2,911152733,2020,990,501(c)(2),10,0.363716,0.0,0,5794.0,NaN,1420 NE NORTHGATE WAY SEATTLE WA 98125,0,2
2,3,223519265,2020,990,501(c)(3),41,0.072674,1.0,0,25534.0,NaN,713 WASHINGTON STREET HOBOKEN NJ 07030,0,1
3,4,260741074,2020,990EZ,501(c)(3),26,NaN,NaN,0,28827.0,NaN,65 Glen Road Box 246 Garner NC 27529,0,1
4,5,814522819,2020,990EZ,501(c)(3),3,NaN,NaN,0,85049.0,NaN,1610 TODDS LN Hampton VA 23666,0,1


A last feature that could indicate similar behavior/intentions would be if the address of the organization has changed frequently over time.

In [57]:
features = features.merge(
    features.groupby("ein",as_index=False).agg(address_changes=("address", "nunique")),
    on="ein",
    how="left"
)
features = features.drop("address", axis=1)
features.head()

,filing_id,ein,tax_year,form_type,exempt_organization_type,mission,grants_as_rev_frac,perc_ind_voting_members,political_activity_flag,total_expenses,lobbying_expenses,dba_changes,name_changes,address_changes
0,1,272292010,2019,990,501(c)(3),32,0.673025,1.0,0,1137870.0,NaN,0,1,2
1,2,911152733,2020,990,501(c)(2),10,0.363716,0.0,0,5794.0,NaN,0,2,2
2,3,223519265,2020,990,501(c)(3),41,0.072674,1.0,0,25534.0,NaN,0,1,1
3,4,260741074,2020,990EZ,501(c)(3),26,NaN,NaN,0,28827.0,NaN,0,1,2
4,5,814522819,2020,990EZ,501(c)(3),3,NaN,NaN,0,85049.0,NaN,0,1,1


The fraction of total expenses geared towards lobbying could also correlate with dark money organizations.

In [58]:
features["frac_spending_lob"] = features.apply(lambda row: row.lobbying_expenses / row.total_expenses if pd.notna(row.lobbying_expenses) and row.total_expenses > 0 else 0, axis=1)
features = features.drop(["total_expenses", "lobbying_expenses"], axis=1)
features.head()

,filing_id,ein,tax_year,form_type,exempt_organization_type,mission,grants_as_rev_frac,perc_ind_voting_members,political_activity_flag,dba_changes,name_changes,address_changes,frac_spending_lob
0,1,272292010,2019,990,501(c)(3),32,0.673025,1.0,0,0,1,2,0.0
1,2,911152733,2020,990,501(c)(2),10,0.363716,0.0,0,0,2,2,0.0
2,3,223519265,2020,990,501(c)(3),41,0.072674,1.0,0,0,1,1,0.0
3,4,260741074,2020,990EZ,501(c)(3),26,NaN,NaN,0,0,1,2,0.0
4,5,814522819,2020,990EZ,501(c)(3),3,NaN,NaN,0,0,1,1,0.0


Impute missing values with zeros. If the filer is missing data that would cause the org to note have the data, then it can be treated as a zero. This is not a perfect solution, but more informed imputing could be included in future projects.

In [59]:
features = features.fillna(0)
features.head()

,filing_id,ein,tax_year,form_type,exempt_organization_type,mission,grants_as_rev_frac,perc_ind_voting_members,political_activity_flag,dba_changes,name_changes,address_changes,frac_spending_lob
0,1,272292010,2019,990,501(c)(3),32,0.673025,1.0,0,0,1,2,0.0
1,2,911152733,2020,990,501(c)(2),10,0.363716,0.0,0,0,2,2,0.0
2,3,223519265,2020,990,501(c)(3),41,0.072674,1.0,0,0,1,1,0.0
3,4,260741074,2020,990EZ,501(c)(3),26,0.000000,0.0,0,0,1,2,0.0
4,5,814522819,2020,990EZ,501(c)(3),3,0.000000,0.0,0,0,1,1,0.0


In [60]:
df_to_s3(features, path="parquet_updated/modeling_features.parquet")